
将源代码 parsing 成一个 parse tree。

将 parse tree 转化为 AST（抽象语法树， abstract syntax tree）。

生成符号表（symbol table）。

从 AST 生成 code object，这一步包含：

将 AST 转化为一个 flow control graph。

从 control flow graph 产生 code object。

In [62]:
def example():
    a=1
    b=a+1
    return b
src=b"""
def example():
    a=1
    b=a+1
    if a >1:
        pass
    return b
"""

In [63]:
import tokenize
from io import BytesIO

for tok in tokenize.tokenize(BytesIO(src).readline):
    print(tok)

TokenInfo(type=67 (ENCODING), string='utf-8', start=(0, 0), end=(0, 0), line='')
TokenInfo(type=65 (NL), string='\n', start=(1, 0), end=(1, 1), line='\n')
TokenInfo(type=1 (NAME), string='def', start=(2, 0), end=(2, 3), line='def example():\n')
TokenInfo(type=1 (NAME), string='example', start=(2, 4), end=(2, 11), line='def example():\n')
TokenInfo(type=55 (OP), string='(', start=(2, 11), end=(2, 12), line='def example():\n')
TokenInfo(type=55 (OP), string=')', start=(2, 12), end=(2, 13), line='def example():\n')
TokenInfo(type=55 (OP), string=':', start=(2, 13), end=(2, 14), line='def example():\n')
TokenInfo(type=4 (NEWLINE), string='\n', start=(2, 14), end=(2, 15), line='def example():\n')
TokenInfo(type=5 (INDENT), string='    ', start=(3, 0), end=(3, 4), line='    a=1\n')
TokenInfo(type=1 (NAME), string='a', start=(3, 4), end=(3, 5), line='    a=1\n')
TokenInfo(type=55 (OP), string='=', start=(3, 5), end=(3, 6), line='    a=1\n')
TokenInfo(type=2 (NUMBER), string='1', start=(3, 6),

In [65]:
# import parser
# from pprint import pprint
# code_str = """
# def hello_world():
#     return 'hello world'
# """
# st = parser.suite(code_str)
# pprint(parser.st2list(st)) #3.9后不再对外暴露parser tree， 因为改用peg了

In [21]:
import ast

tree = ast.parse(src)

print(ast.dump(tree, indent=4))

Module(
    body=[
        FunctionDef(
            name='example',
            args=arguments(
                posonlyargs=[],
                args=[],
                kwonlyargs=[],
                kw_defaults=[],
                defaults=[]),
            body=[
                Assign(
                    targets=[
                        Name(id='a', ctx=Store())],
                    value=Constant(value=1)),
                Assign(
                    targets=[
                        Name(id='b', ctx=Store())],
                    value=BinOp(
                        left=Name(id='a', ctx=Load()),
                        op=Add(),
                        right=Constant(value=1))),
                Return(
                    value=Name(id='b', ctx=Load()))],
            decorator_list=[],
            type_params=[])],
    type_ignores=[])


In [ ]:
%pip install instaviz
import instaviz
instaviz.show(example)

In [3]:
dir(example)

['__annotations__',
 '__builtins__',
 '__call__',
 '__class__',
 '__closure__',
 '__code__',
 '__defaults__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__get__',
 '__getattribute__',
 '__getstate__',
 '__globals__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__kwdefaults__',
 '__le__',
 '__lt__',
 '__module__',
 '__name__',
 '__ne__',
 '__new__',
 '__qualname__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__type_params__']

In [17]:
import dis
dis.dis(example)

  1           0 RESUME                   0

  2           2 LOAD_GLOBAL              1 (NULL + print)
             12 LOAD_GLOBAL              2 (inspect)
             22 LOAD_ATTR                5 (NULL|self + currentframe)
             42 CALL                     0
             50 CALL                     1
             58 POP_TOP

  3          60 LOAD_CONST               1 (1)
             62 STORE_FAST               0 (a)

  4          64 LOAD_FAST                0 (a)
             66 LOAD_CONST               1 (1)
             68 BINARY_OP                0 (+)
             72 STORE_FAST               1 (b)

  5          74 LOAD_FAST                1 (b)
             76 RETURN_VALUE


In [24]:
%pip install bytecode

Note: you may need to restart the kernel to use updated packages.


In [25]:
from bytecode import Bytecode, ControlFlowGraph

bc = Bytecode.from_code(example.__code__)
cfg = ControlFlowGraph.from_bytecode(bc)
print("ByteCode",bc)
for i, block in enumerate(cfg):
    print("BLOCK", i)
    for instr in block:
        print(" ", instr)

ByteCode [<RESUME arg=0 location=InstrLocation(lineno=1, end_lineno=1, col_offset=0, end_col_offset=0)>, <LOAD_CONST arg=1 location=InstrLocation(lineno=2, end_lineno=2, col_offset=6, end_col_offset=7)>, <STORE_FAST arg='a' location=InstrLocation(lineno=2, end_lineno=2, col_offset=4, end_col_offset=5)>, <LOAD_FAST arg='a' location=InstrLocation(lineno=3, end_lineno=3, col_offset=6, end_col_offset=7)>, <LOAD_CONST arg=1 location=InstrLocation(lineno=3, end_lineno=3, col_offset=8, end_col_offset=9)>, <BINARY_OP arg=<BinaryOp.ADD: 0> location=InstrLocation(lineno=3, end_lineno=3, col_offset=6, end_col_offset=9)>, <STORE_FAST arg='b' location=InstrLocation(lineno=3, end_lineno=3, col_offset=4, end_col_offset=5)>, <LOAD_FAST arg='b' location=InstrLocation(lineno=4, end_lineno=4, col_offset=11, end_col_offset=12)>, <RETURN_VALUE location=InstrLocation(lineno=4, end_lineno=4, col_offset=4, end_col_offset=12)>]
BLOCK 0
  <RESUME arg=0 location=InstrLocation(lineno=1, end_lineno=1, col_offset=0

In [6]:
import inspect
print(inspect.getdoc(inspect))

Get useful information from live Python objects.

This module encapsulates the interface provided by the internal special
attributes (co_*, im_*, tb_*, etc.) in a friendlier fashion.
It also provides some help for examining source code and class layout.

Here are some of the useful functions provided by this module:

    ismodule(), isclass(), ismethod(), isfunction(), isgeneratorfunction(),
        isgenerator(), istraceback(), isframe(), iscode(), isbuiltin(),
        isroutine() - check object types
    getmembers() - get members of an object that satisfy a given condition

    getfile(), getsourcefile(), getsource() - find an object's source code
    getdoc(), getcomments() - get documentation on an object
    getmodule() - determine the module that an object came from
    getclasstree() - arrange classes so as to represent their hierarchy

    getargvalues(), getcallargs() - get info about function arguments
    getfullargspec() - same, with support for Python 3 features
    forma

In [8]:
print(inspect.currentframe())

<frame at 0x103ff2fb0, file '/var/folders/_1/bny2r6_91qv_7_6fdv2w9cy80000gn/T/ipykernel_9149/398361378.py', line 1, code <module>>


In [9]:
def example():
    print(inspect.currentframe())
    a=1
    b=a+1
    return b
example()

<frame at 0x104b9cd00, file '/var/folders/_1/bny2r6_91qv_7_6fdv2w9cy80000gn/T/ipykernel_9149/4032919443.py', line 2, code example>


2

In [11]:
frame=inspect.currentframe()
dir(frame)

['__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'clear',
 'f_back',
 'f_builtins',
 'f_code',
 'f_globals',
 'f_lasti',
 'f_lineno',
 'f_locals',
 'f_trace',
 'f_trace_lines',
 'f_trace_opcodes']

In [27]:
import ctypes

class PyObject(ctypes.Structure):
    _fields_ = [
        ("ob_refcnt", ctypes.c_ssize_t),
        ("ob_type", ctypes.c_void_p),
    ]

x = 1000000
obj = PyObject.from_address(id(x))

print("object:", hex(id(x)))
print("refcnt:", obj.ob_refcnt)
print("ob_type:", hex(obj.ob_type))
print("type addr:", hex(id(type(x))))

object: 0x10504c5b0
refcnt: 2
ob_type: 0x100c3b7b0
type addr: 0x100c3b7b0


In [42]:
%pip install snoop

Note: you may need to restart the kernel to use updated packages.


In [43]:
import snoop

@snoop
def number_to_bits(number):
    if number:
        bits = []
        while number:
            number, remainder = divmod(number, 2)
            bits.insert(0, remainder)
        return bits
    else:
        return [0]

number_to_bits(6)

/opt/homebrew/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/homebrew/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (
15:59:11.98 >>> Call to number_to_bits in File "/var/folders/_1/bny2r6_91qv_7_6fdv2w9cy80000gn/T/ipykernel_9149/903065405.py", line 4
15:59:11.98 ...... number = 6
15:59:11.98    4 | def number_to_bits(number):
15:59:11.98    5 |     if number:
15:59:11.98    6 |         bits = []
15:59:11.98    7 |         while number:
15:59:11.98    8 |             number, remainder = divmod(number, 2)
15:59:11.98 .................. number = 3
15:59:11.98 .................. remainder = 0
15:59:11.98    9 |             bi

[1, 1, 0]

In [30]:
from collections import Counter
lites = ['astropy__astropy-12699\n', 'astropy__astropy-13471\n', 'astropy__astropy-13898\n', 'astropy__astropy-15900\n', 'astropy__astropy-16295\n', 'astropy__astropy-7010\n', 'astropy__astropy-7422\n', 'astropy__astropy-7616\n', 'dask__dask-10356\n', 'dask__dask-10428\n', 'dask__dask-5501\n', 'dask__dask-5553\n', 'matplotlib__matplotlib-17995\n', 'matplotlib__matplotlib-22108\n', 'numpy__numpy-12575\n', 'numpy__numpy-18324\n', 'numpy__numpy-21832\n', 'numpy__numpy-25788\n', 'numpy__numpy-27830\n', 'pandas-dev__pandas-23888\n', 'pandas-dev__pandas-24083\n', 'pandas-dev__pandas-24308\n', 'pandas-dev__pandas-25953\n', 'pandas-dev__pandas-26702\n', 'pandas-dev__pandas-27448\n', 'pandas-dev__pandas-29820\n', 'pandas-dev__pandas-32825\n', 'pandas-dev__pandas-34178\n', 'pandas-dev__pandas-34192\n', 'pandas-dev__pandas-36638\n', 'pandas-dev__pandas-37064\n', 'pandas-dev__pandas-38560\n', 'pandas-dev__pandas-39332\n', 'pandas-dev__pandas-39972\n', 'pandas-dev__pandas-40818\n', 'pandas-dev__pandas-42353\n', 'pandas-dev__pandas-42704\n', 'pandas-dev__pandas-42841\n', 'pandas-dev__pandas-43073\n', 'pandas-dev__pandas-43243\n', 'pandas-dev__pandas-43285\n', 'pandas-dev__pandas-43335\n', 'pandas-dev__pandas-43352\n', 'pandas-dev__pandas-43518\n', 'pandas-dev__pandas-43683\n', 'pandas-dev__pandas-43694\n', 'pandas-dev__pandas-44566\n', 'pandas-dev__pandas-44594\n', 'pandas-dev__pandas-44827\n', 'pandas-dev__pandas-44908\n', 'pandas-dev__pandas-45387\n', 'pandas-dev__pandas-46040\n', 'pandas-dev__pandas-46109\n', 'pandas-dev__pandas-46349\n', 'pandas-dev__pandas-48152\n', 'pandas-dev__pandas-48723\n', 'pandas-dev__pandas-48752\n', 'pandas-dev__pandas-49177\n', 'pandas-dev__pandas-49772\n', 'pandas-dev__pandas-49839\n', 'pandas-dev__pandas-50078\n', 'pandas-dev__pandas-50168\n', 'pandas-dev__pandas-50620\n', 'pandas-dev__pandas-51518\n', 'pandas-dev__pandas-52430\n', 'pandas-dev__pandas-52525\n', 'pandas-dev__pandas-52672\n', 'pandas-dev__pandas-52836\n', 'pandas-dev__pandas-53088\n', 'pandas-dev__pandas-53152\n', 'pandas-dev__pandas-53731\n', 'pandas-dev__pandas-54299\n', 'pandas-dev__pandas-55084\n', 'pandas-dev__pandas-56089\n', 'pandas-dev__pandas-56110\n', 'pandas-dev__pandas-56128\n', 'pandas-dev__pandas-56345\n', 'pandas-dev__pandas-56508\n', 'pandas-dev__pandas-56806\n', 'pandas-dev__pandas-57459\n', 'pandas-dev__pandas-57479\n', 'pandas-dev__pandas-57855\n', 'pydata__xarray-5661\n', 'pydata__xarray-7382\n', 'pydata__xarray-9001\n', 'scikit-learn__scikit-learn-15049\n', 'scikit-learn__scikit-learn-17235\n', 'scikit-learn__scikit-learn-9843\n', 'scipy__scipy-10393\n', 'scipy__scipy-10467\n', 'scipy__scipy-10939\n', 'scipy__scipy-11358\n', 'scipy__scipy-11517\n', 'scipy__scipy-18917\n', 'scipy__scipy-22660\n', 'sympy__sympy-11675\n', 'sympy__sympy-15379\n', 'sympy__sympy-16134\n', 'sympy__sympy-20989\n', 'sympy__sympy-21391\n']
cnt = Counter()
for d in lites:
    cnt[ d.split('__')[0] ]+=1
cnt

Counter({'pandas-dev': 63,
         'astropy': 8,
         'scipy': 7,
         'numpy': 5,
         'sympy': 5,
         'dask': 4,
         'pydata': 3,
         'scikit-learn': 3,
         'matplotlib': 2})

In [38]:
nf=[instance for instance in """
dask__dask-10922
pydata__xarray-5661
pydata__xarray-7382
numpy__numpy-11720
dask__dask-11625
astropy__astropy-13471
sympy__sympy-10919
pandas-dev__pandas-24083
astropy__astropy-12699
pandas-dev__pandas-24308
pydata__xarray-7472
matplotlib__matplotlib-15346
scikit-learn__scikit-learn-15049
sympy__sympy-10621
dask__dask-10428
pandas-dev__pandas-24023
pandas-dev__pandas-23888
scipy__scipy-10064
pandas-dev__pandas-23772
numpy__numpy-12575
scikit-learn__scikit-learn-13310
pydata__xarray-4740
dask__dask-5501
matplotlib__matplotlib-13917
scipy__scipy-10467
scikit-learn__scikit-learn-13290
matplotlib__matplotlib-17177
scipy__scipy-10564
scikit-learn__scikit-learn-15257
scipy__scipy-10393
scipy__scipy-10477
astropy__astropy-13497
pydata__xarray-7374
numpy__numpy-12321
astropy__astropy-12701
numpy__numpy-12596
numpy__numpy-13250
""".split("\n") if instance]
lites = [instance for instance in """
astropy__astropy-12699
astropy__astropy-13471
astropy__astropy-13898
astropy__astropy-15900
astropy__astropy-16295
astropy__astropy-7010
astropy__astropy-7422
astropy__astropy-7616
dask__dask-10356
dask__dask-10428
dask__dask-5501
dask__dask-5553
matplotlib__matplotlib-17995
matplotlib__matplotlib-22108
numpy__numpy-12575
numpy__numpy-18324
numpy__numpy-21832
numpy__numpy-25788
numpy__numpy-27830
pandas-dev__pandas-23888
pandas-dev__pandas-24083
pandas-dev__pandas-24308
pandas-dev__pandas-25953
pandas-dev__pandas-26702
pandas-dev__pandas-27448
pandas-dev__pandas-29820
pandas-dev__pandas-32825
pandas-dev__pandas-34178
pandas-dev__pandas-34192
pandas-dev__pandas-36638
pandas-dev__pandas-37064
pandas-dev__pandas-38560
pandas-dev__pandas-39332
pandas-dev__pandas-39972
pandas-dev__pandas-40818
pandas-dev__pandas-42353
pandas-dev__pandas-42704
pandas-dev__pandas-42841
pandas-dev__pandas-43073
pandas-dev__pandas-43243
pandas-dev__pandas-43285
pandas-dev__pandas-43335
pandas-dev__pandas-43352
pandas-dev__pandas-43518
pandas-dev__pandas-43683
pandas-dev__pandas-43694
pandas-dev__pandas-44566
pandas-dev__pandas-44594
pandas-dev__pandas-44827
pandas-dev__pandas-44908
pandas-dev__pandas-45387
pandas-dev__pandas-46040
pandas-dev__pandas-46109
pandas-dev__pandas-46349
pandas-dev__pandas-48152
pandas-dev__pandas-48723
pandas-dev__pandas-48752
pandas-dev__pandas-49177
pandas-dev__pandas-49772
pandas-dev__pandas-49839
pandas-dev__pandas-50078
pandas-dev__pandas-50168
pandas-dev__pandas-50620
pandas-dev__pandas-51518
pandas-dev__pandas-52430
pandas-dev__pandas-52525
pandas-dev__pandas-52672
pandas-dev__pandas-52836
pandas-dev__pandas-53088
pandas-dev__pandas-53152
pandas-dev__pandas-53731
pandas-dev__pandas-54299
pandas-dev__pandas-55084
pandas-dev__pandas-56089
pandas-dev__pandas-56110
pandas-dev__pandas-56128
pandas-dev__pandas-56345
pandas-dev__pandas-56508
pandas-dev__pandas-56806
pandas-dev__pandas-57459
pandas-dev__pandas-57479
pandas-dev__pandas-57855
pydata__xarray-5661
pydata__xarray-7382
pydata__xarray-9001
scikit-learn__scikit-learn-15049
scikit-learn__scikit-learn-17235
scikit-learn__scikit-learn-9843
scipy__scipy-10393
scipy__scipy-10467
scipy__scipy-10939
scipy__scipy-11358
scipy__scipy-11517
scipy__scipy-18917
scipy__scipy-22660
sympy__sympy-11675
sympy__sympy-15379
sympy__sympy-16134
sympy__sympy-20989
sympy__sympy-21391
""".split("\n") if instance]

set(nf) in set(lites) # False
set(nf) - set(lites)

{'astropy__astropy-12701',
 'astropy__astropy-13497',
 'dask__dask-10922',
 'dask__dask-11625',
 'matplotlib__matplotlib-13917',
 'matplotlib__matplotlib-15346',
 'matplotlib__matplotlib-17177',
 'numpy__numpy-11720',
 'numpy__numpy-12321',
 'numpy__numpy-12596',
 'numpy__numpy-13250',
 'pandas-dev__pandas-23772',
 'pandas-dev__pandas-24023',
 'pydata__xarray-4740',
 'pydata__xarray-7374',
 'pydata__xarray-7472',
 'scikit-learn__scikit-learn-13290',
 'scikit-learn__scikit-learn-13310',
 'scikit-learn__scikit-learn-15257',
 'scipy__scipy-10064',
 'scipy__scipy-10477',
 'scipy__scipy-10564',
 'sympy__sympy-10621',
 'sympy__sympy-10919'}

In [61]:
"""
""".count("PREEDIT")


36